[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/arap/arap.ipynb)

# As Rigid As Possible

A bar is held at one end and its other end is bent a quarter turn up. Every other vertex goes where its neighbourhood stays as close as it can to a turn of its shape at rest: the surface bends without shearing or shrinking. [Sorkine and Alexa (2007)](https://igl.ethz.ch/projects/ARAP/arap_web.pdf) find it by alternating two steps, the best rotor at each vertex and the best vertices for those rotors. Here the rotors come from a form with the rotor left open, and every operator is a *sparse extensor*: a linear map between fields on the mesh's vertices and edges, which couples each element to a few others.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import stack
from examples.surfaces.arap import core, render
from examples.surfaces.arap.core import Even, Vector, mv
from examples.surfaces.spin_transformations.core import as_diag, as_ga_sparse, as_scalar

# The bar: its squares per side of the cube it is stretched from, and its length.
DIVISIONS = 10
LENGTH = 6.0
# How stiffly the handles are held, how far the far end turns, and the frames over the bend with the
# iterations at each, each frame starting from the last.
STIFFNESS = 1e3
BEND = np.pi / 2
FRAMES = 24
ITERATIONS = 6
# How near its end a vertex is held, and the checkerboard's cell, one square of the bar at rest.
MARGIN = 1e-6
CELL = np.array([LENGTH, 2.0, 2.0]) / DIVISIONS

bar = core.bar(DIVISIONS, LENGTH)
render.side_by_side(bar, [bar.vertices], ["the bar at rest"], CELL);

## 1. The handles

The vertices at both ends are the handles. The near end stays where it is, and the far end is carried along an arc of the bar's length, turned by the rotor of the angle about y, a little further at each frame. Each pose says where every vertex would be with only the far end moved; only the handles' part of it counts.

In [ ]:
along = bar.vertices | mv.x                                                # Scalar[V]
near, far = along < -LENGTH / 2 + MARGIN, along > LENGTH / 2 - MARGIN      # [V] boolean
handles = near | far                                                       # [V] boolean
moved = as_scalar(far).field()                                             # Scalar[V]


def bent(vertices: Vector, angle: float) -> Vector:
    """The far end of the bar where an arc of its length would carry it, turned by the angle about y:
    its centre on the arc from the near end, curving up."""
    radius = LENGTH / angle
    centre = mv.x * (-LENGTH / 2 + radius * np.sin(angle)) + mv.z * (radius * (1 - np.cos(angle)))   # [] Vector
    turn = ((mv.x ^ mv.z) * (-angle / 2)).exp()                            # [] Rotor
    return centre + (turn >> (vertices - mv.x * (LENGTH / 2)))


angles = np.linspace(0.0, BEND, FRAMES + 1)[1:]                            # [frames]
poses = [bar.vertices + moved * (bent(bar.vertices, angle) - bar.vertices) for angle in angles]   # [frames] Vector[V]
render.side_by_side(bar, [poses[-1]], ["the far end moved"], CELL);

## 2. Laplacian editing

The bar's edges, `T10 * bar.vertices`, are a field with one vector per edge. `T10` is a sparse extensor from vertex fields to edge fields: each edge couples to its two ends through the scalars minus one and one, so it takes each edge's tail from its head. `A10` averages each edge's two ends, `H1` weighs each edge by its cotangent weight, and `P` holds the handles by a stiff penalty. `~T10` is its reverse: reversing a product turns it around, so every coupling runs the other way, from an edge to its two ends. The edges' energy weighs each edge against its own reverse, `~edges * H1 * edges` with `edges = T10 * vertices`, and with the vertices left open on both sides that is `~T10 * H1 * T10`, the cotangent Laplacian.

Its couplings are scalars, and the product with an open type, `* Vector`, makes it a map of vector fields into themselves, which `solve` can invert. The vertices whose edges best match the rest edges as they are, with the handles moved, are Laplacian editing: the bar bends, but shears and shrinks where it does.

In [ ]:
# the boundary, taking each edge's tail from its head, and the mean over each edge
I10 = bar.edges                                                            # [E, 2]
T10 = as_ga_sparse(I10, as_scalar(np.ones_like(I10) * [-1, 1]))           # [E, V] Scalar
A10 = as_ga_sparse(I10, as_scalar(np.ones_like(I10) / 2))                 # [E, V] Scalar

# diagonal operators: each edge's cotangent weight, and the handles' stiffness
H1 = as_diag(bar.edge_ratio)                                               # [E, E] Scalar
P = as_diag(as_scalar(handles * STIFFNESS).field())                        # [V, V] Scalar

rest = T10 * bar.vertices                                                  # Vector[E]
# the cotangent Laplacian, with the handles held
held = (~T10 * H1 * T10 + P) * Vector                                      # Vector[V] <- Vector[V]
# the vertices whose edges best match the rest edges as they are
laplacian = stack([held.solve(~T10 * H1 * rest + P * targets) for targets in poses])   # [frames] Vector[V]

print(T10.gatype, T10.shape)
print(held.gatype, held.shape)
render.side_by_side(bar, [laplacian[-1]], ["Laplacian editing"], CELL);

## 3. The best rotor at each vertex

Given where the vertices are, each vertex's best rotor `R` turns its rest edges onto its edges as they are: it makes the most of `edge | (R >> rest)`, summed over the vertex's edges. With the rotor left open, `Even >> rest` is the rest edge turned by any even multivector, a map `Vector <- (Even, Even)` that holds the rotor in both places the sandwich does. Paired with the edge it is a form `Scalar <- (Even, Even)`, a number for every two rotors, with `form(R, R) == edge | (R >> rest)`. The forms are a field like any other, so `~A10 * H1` sums each vertex's edge forms, weighted, as it would sum numbers. The rotor that makes the most of each sum is its top eigenvector, so one batched `eigh` finds every vertex's rotor at once.

In the notation of Sorkine and Alexa the form reads as the trace of $R_i S_i$, with the covariance $S_i = \sum_j w_{ij}\, e_{ij} {e'_{ij}}^{\top}$.

In [ ]:
vertices = laplacian[-1]                                                   # Vector[V]
edges = T10 * vertices                                                     # Vector[E]
# each edge's form on a rotor open twice, summed at each vertex over its edges
forms = ~A10 * H1 * (edges | (Even >> rest))                               # Scalar[V] <- (Even, Even)
# each vertex's best rotor: the top eigenvector of its form
rotors = forms.batch().eigh()[1][..., -1].field()                                  # Even[V]

print((Even >> rest).gatype)
print(forms.gatype)

## 4. The best vertices for the rotors

A rotor in the sandwich with the vector left open, `R >> Vector`, is the turn itself, a map `Vector <- Vector`. `A10` averages the two turns at each edge's ends as maps, and the averaged map then takes the rest edge. Averaging maps rather than rotors does not depend on the sign the eigensolver gave each rotor: `R >> Vector` and `-R >> Vector` are one map. The vertices that best match the turned edges come from the same solve as Laplacian editing, with the turned edges in place of the rest edges. One step on, the bar already keeps more of its shape.

In [ ]:
# each rest edge turned by its two ends' rotors, the turns averaged as maps
turned = (A10 * (rotors >> Vector))(rest)                                  # Vector[E]
# the vertices whose edges best match the turned ones
stepped = held.solve(~T10 * H1 * turned + P * poses[-1])                   # Vector[V]

render.side_by_side(bar, [laplacian[-1], stepped], ["Laplacian editing", "one step on"], CELL);

## 5. Bending

Each step lowers the energy, the cotangent-weighted squared mismatch between every edge and its rest edge turned by its two ends' rotors, so alternating them settles. A rotor keeps lengths, so an edge's mismatch at its two ends adds up to `2 * (edge**2 + rest**2 - 2 * edge | turned)`. Each frame starts from the last one's vertices, with the handles moved a little further.

In the notation of Sorkine and Alexa the energy reads as $E = \sum_i \sum_{j \in N(i)} w_{ij} \|(p'_i - p'_j) - R_i (p_i - p_j)\|^2$.

In [ ]:
def bend(poses: list[Vector], iterations: int):
    """For each pose of the handles, the vertices after the given iterations of the two steps, each pose
    starting from the last one's, and the energy at each iteration."""
    vertices = bar.vertices                                                # Vector[V]
    for targets in poses:
        energies = []
        for _ in range(iterations):
            edges = T10 * vertices                                         # Vector[E]
            forms = ~A10 * H1 * (edges | (Even >> rest))                   # Scalar[V] <- (Even, Even)
            rotors = forms.batch().eigh()[1][..., -1].field()                      # Even[V]
            turned = (A10 * (rotors >> Vector))(rest)                      # Vector[E]
            mismatch = H1 * (edges.norm_squared() + rest.norm_squared() - 2 * (edges | turned))   # Scalar[E]
            penalty = P * (vertices - targets).norm_squared()              # Scalar[V]
            energies.append(2 * mismatch.batch().sum(axis=-1) + penalty.batch().sum(axis=-1))
            vertices = held.solve(~T10 * H1 * turned + P * targets)        # Vector[V]
        yield vertices, stack(energies)


bent_frames = list(bend(poses, ITERATIONS))
rigid = stack([vertices for vertices, _ in bent_frames])                   # [frames] Vector[V]
energies = bent_frames[-1][1]                                              # [iterations] Scalar
render.side_by_side(bar, [rigid[-1], laplacian[-1]], ["as rigid as possible", "Laplacian editing"], CELL)
render.inline(render.animate(bar, rigid, CELL), 80)

In [ ]:
# checks
# The iteration laid out here is the core's.
reference = stack([vertices for vertices, _, _ in core.deform(bar, handles, poses, STIFFNESS, ITERATIONS)])
np.testing.assert_allclose((reference - rigid).kernel, 0.0, atol=1e-9)
# The form evaluated at a rotor is the edge against the rest edge it turns.
form = edges | (Even >> rest)                                              # Scalar[E] <- (Even, Even)
rotor = rotors.batch()[0]                                                  # [] Even
np.testing.assert_allclose((form(rotor, rotor) - (edges | (rotor >> rest))).kernel, 0.0, atol=1e-12)
# A rotor and its negative make one turn.
np.testing.assert_array_equal((rotors >> Vector).kernel, (-rotors >> Vector).kernel)
# At the last frame the energy falls with every iteration, and the handles are where they were moved to.
falling = energies.to_array()
assert np.all(np.diff(falling) <= 1e-9 * falling[0])
assert (rigid[-1] - poses[-1]).norm().batch()[handles].to_array().max() < 1e-3
# The bar keeps its angles: its corners' cosines change a fraction as much as by Laplacian editing.
change = [np.abs((bar.copy(vertices=shape).corner_cosines() - bar.corner_cosines()).to_array()).mean()
          for shape in (rigid[-1], laplacian[-1])]
assert change[0] < 0.5 * change[1]